# Day 8 — Silver Layer: API Data Transformation
**Bronze Volume JSON → Silver Delta Tables (all 17 entities)**

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/api/<entity>/ingestion_date=<YYYY-MM-DD>/page_N.json`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api/<entity>/` (Delta format)

**What this notebook does:**
- Reads all Bronze JSON pages for each entity (all ingestion dates or a specific date)
- Flattens the `data` array from the API pagination wrapper
- Casts columns to correct types (timestamps, integers, decimals)
- Adds Silver audit columns: `silver_ingested_at`, `silver_load_type`, `silver_pipeline`
- Deduplicates on the natural key (keeps latest `updated_at` per key)
- Writes to Silver as Delta tables (overwrite for full, merge/upsert for incremental)
- Produces a run summary with row counts and status per entity

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    DoubleType, DecimalType, TimestampType, BooleanType, DateType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone
import json

print("Imports OK")

In [ ]:
# Cell 2 — Widget: load_type
dbutils.widgets.removeAll()
dbutils.widgets.text("load_type", "incremental", "Load Type (full / incremental)")
dbutils.widgets.text("ingestion_date", "", "Ingestion Date (YYYY-MM-DD, blank = latest)")

LOAD_TYPE      = dbutils.widgets.get("load_type").strip().lower()
INGESTION_DATE = dbutils.widgets.get("ingestion_date").strip()

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type      : {LOAD_TYPE}")
print(f"ingestion_date : {INGESTION_DATE or '(latest available)'}")

In [ ]:
# Cell 3 — Constants
# Both Bronze and Silver use the same 'default' schema — silver-volume is a separate Volume
# under the same catalog/schema as bronze-volume
BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
SILVER_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume"

BRONZE_API_BASE = f"{BRONZE_VOLUME}/api"
SILVER_API_BASE = f"{SILVER_VOLUME}/api"

PIPELINE_NAME   = "pl_silver_api_transformation_v1"
RUN_TS          = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"BRONZE_API_BASE : {BRONZE_API_BASE}")
print(f"SILVER_API_BASE : {SILVER_API_BASE}")
print(f"RUN_TS          : {RUN_TS}")

In [ ]:
# Cell 4 — Entity schema registry
# Each entry: (natural_key, cdc_field, schema_dict)
# schema_dict maps column_name -> spark type string used in CAST

ENTITY_REGISTRY = {
    "payments": {
        "natural_key" : "payment_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "payment_id"    : "string",
            "session_id"    : "string",
            "customer_id"   : "string",
            "amount"        : "decimal(10,2)",
            "currency"      : "string",
            "status"        : "string",
            "payment_method": "string",
            "created_at"    : "timestamp",
            "updated_at"    : "timestamp",
        }
    },
    "sessions": {
        "natural_key" : "session_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "session_id"       : "string",
            "vehicle_id"       : "string",
            "charger_id"       : "string",
            "customer_id"      : "string",
            "station_id"       : "string",
            "start_time"       : "timestamp",
            "end_time"         : "timestamp",
            "energy_kwh"       : "decimal(10,4)",
            "duration_minutes" : "integer",
            "status"           : "string",
            "created_at"       : "timestamp",
            "updated_at"       : "timestamp",
        }
    },
    "customers": {
        "natural_key" : "customer_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "customer_id"    : "string",
            "first_name"     : "string",
            "last_name"      : "string",
            "email"          : "string",
            "phone"          : "string",
            "city"           : "string",
            "state"          : "string",
            "country"        : "string",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "fleet": {
        "natural_key" : "vehicle_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "vehicle_id"        : "string",
            "make"              : "string",
            "model"             : "string",
            "year"              : "integer",
            "battery_capacity"  : "decimal(8,2)",
            "range_km"          : "decimal(8,2)",
            "status"            : "string",
            "partner_id"        : "string",
            "created_at"        : "timestamp",
            "updated_at"        : "timestamp",
        }
    },
    "chargers": {
        "natural_key" : "charger_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "charger_id"     : "string",
            "station_id"     : "string",
            "charger_type"   : "string",
            "power_kw"       : "decimal(8,2)",
            "status"         : "string",
            "connector_type" : "string",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "vehicles": {
        "natural_key" : "vehicle_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "vehicle_id"       : "string",
            "customer_id"      : "string",
            "make"             : "string",
            "model"            : "string",
            "year"             : "integer",
            "battery_capacity" : "decimal(8,2)",
            "range_km"         : "decimal(8,2)",
            "license_plate"    : "string",
            "created_at"       : "timestamp",
            "updated_at"       : "timestamp",
        }
    },
    "stations": {
        "natural_key" : "station_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "station_id"   : "string",
            "name"         : "string",
            "city"         : "string",
            "state"        : "string",
            "country"      : "string",
            "latitude"     : "decimal(10,7)",
            "longitude"    : "decimal(10,7)",
            "total_chargers": "integer",
            "status"       : "string",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "complaints": {
        "natural_key" : "complaint_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "complaint_id"   : "string",
            "customer_id"    : "string",
            "session_id"     : "string",
            "category"       : "string",
            "description"    : "string",
            "status"         : "string",
            "priority"       : "string",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "maintenance_events": {
        "natural_key" : "event_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "event_id"       : "string",
            "charger_id"     : "string",
            "station_id"     : "string",
            "event_type"     : "string",
            "description"    : "string",
            "technician_id"  : "string",
            "status"         : "string",
            "scheduled_date" : "date",
            "completed_date" : "date",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "energy_prices": {
        "natural_key" : "price_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "price_id"       : "string",
            "region"         : "string",
            "price_per_kwh"  : "decimal(8,4)",
            "currency"       : "string",
            "effective_from" : "timestamp",
            "effective_to"   : "timestamp",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "tariffs": {
        "natural_key" : "tariff_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "tariff_id"      : "string",
            "name"           : "string",
            "price_per_kwh"  : "decimal(8,4)",
            "price_per_min"  : "decimal(8,4)",
            "currency"       : "string",
            "valid_from"     : "timestamp",
            "valid_to"       : "timestamp",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "charge_cards": {
        "natural_key" : "card_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "card_id"        : "string",
            "customer_id"    : "string",
            "card_number"    : "string",
            "status"         : "string",
            "issued_at"      : "timestamp",
            "expires_at"     : "timestamp",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "employees": {
        "natural_key" : "employee_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "employee_id"  : "string",
            "first_name"   : "string",
            "last_name"    : "string",
            "email"        : "string",
            "role"         : "string",
            "department"   : "string",
            "station_id"   : "string",
            "hire_date"    : "date",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "partners": {
        "natural_key" : "partner_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "partner_id"   : "string",
            "name"         : "string",
            "country"      : "string",
            "contact_email": "string",
            "status"       : "string",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "cities": {
        "natural_key" : "city_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "city_id"      : "string",
            "name"         : "string",
            "state_code"   : "string",
            "country"      : "string",
            "latitude"     : "decimal(10,7)",
            "longitude"    : "decimal(10,7)",
            "population"   : "long",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "states": {
        "natural_key" : "state_code",
        "cdc_field"   : "updated_at",
        "cast": {
            "state_code"   : "string",
            "name"         : "string",
            "country"      : "string",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "weather": {
        "natural_key" : "city_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "city_id"          : "string",
            "temperature_c"    : "decimal(5,2)",
            "humidity_pct"     : "decimal(5,2)",
            "wind_speed_kmh"   : "decimal(6,2)",
            "condition"        : "string",
            "recorded_at"      : "timestamp",
            "created_at"       : "timestamp",
            "updated_at"       : "timestamp",
        }
    },
}

print(f"Entity registry loaded: {len(ENTITY_REGISTRY)} entities")

In [ ]:
# Cell 5 — Helper: resolve Bronze paths to read

def get_bronze_paths(entity_name, ingestion_date):
    """
    Returns the list of JSON file paths to read from Bronze for a given entity.
    - If ingestion_date is provided: reads only that partition.
    - If blank (full load): reads ALL available ingestion_date partitions.
    """
    base = f"{BRONZE_API_BASE}/{entity_name}"
    try:
        dbutils.fs.ls(base)
    except Exception:
        return []

    if ingestion_date:
        partition = f"{base}/ingestion_date={ingestion_date}"
        try:
            files = dbutils.fs.ls(partition)
            return [f.path for f in files if f.path.endswith(".json")]
        except Exception:
            return []
    else:
        # full load — collect all partitions
        all_files = []
        partitions = dbutils.fs.ls(base)
        for p in partitions:
            try:
                files = dbutils.fs.ls(p.path)
                all_files.extend([f.path for f in files if f.path.endswith(".json")])
            except Exception:
                pass
        return all_files


def folder_exists_dbfs(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


print("Path helpers defined")

In [ ]:
# Cell 6 — Core transformation function

def transform_entity(entity_name, config, load_type, ingestion_date):
    """
    Full Bronze -> Silver transformation for one entity.
    Returns a result dict with status, row counts, and error info.
    """
    result = {
        "entity_name"  : entity_name,
        "status"       : "failed",
        "bronze_rows"  : 0,
        "silver_rows"  : 0,
        "error"        : None,
    }

    try:
        natural_key = config["natural_key"]
        cdc_field   = config["cdc_field"]
        cast_map    = config["cast"]
        silver_path = f"{SILVER_API_BASE}/{entity_name}"

        # ── Step 1: Discover Bronze JSON files ────────────────────────────────
        paths = get_bronze_paths(entity_name, ingestion_date)
        if not paths:
            result["status"] = "skipped"
            result["error"]  = "No Bronze JSON files found"
            return result

        # ── Step 2: Read raw JSON (API pagination wrapper) ────────────────────
        # Each file: {"data": [...records...], "pagination": {...}}
        raw_df = spark.read.option("multiline", "true").json(paths)

        # ── Step 3: Explode the `data` array ──────────────────────────────────
        if "data" not in raw_df.columns:
            result["error"] = "Column 'data' not found in Bronze JSON — schema mismatch"
            return result

        exploded_df = raw_df.select(F.explode(F.col("data")).alias("record"))
        flat_df     = exploded_df.select("record.*")
        result["bronze_rows"] = flat_df.count()

        # ── Step 4: Cast columns to correct types ─────────────────────────────
        cast_exprs = []
        for col_name, col_type in cast_map.items():
            if col_name in flat_df.columns:
                cast_exprs.append(F.col(col_name).cast(col_type).alias(col_name))
            else:
                # Column missing in source — fill with null of correct type
                cast_exprs.append(F.lit(None).cast(col_type).alias(col_name))

        # Keep any extra columns not in schema registry (cast to string)
        registry_cols = set(cast_map.keys())
        extra_cols    = [F.col(c).cast("string").alias(c)
                         for c in flat_df.columns if c not in registry_cols]

        typed_df = flat_df.select(cast_exprs + extra_cols)

        # ── Step 5: Add Silver audit columns ──────────────────────────────────
        typed_df = (
            typed_df
            .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
            .withColumn("silver_load_type",   F.lit(load_type))
            .withColumn("silver_pipeline",    F.lit(PIPELINE_NAME))
        )

        # ── Step 6: Deduplicate on natural key (keep latest updated_at) ───────
        window = Window.partitionBy(natural_key).orderBy(F.col(cdc_field).desc())
        deduped_df = (
            typed_df
            .withColumn("_row_num", F.row_number().over(window))
            .filter(F.col("_row_num") == 1)
            .drop("_row_num")
        )

        # ── Step 7: Write to Silver Delta ──────────────────────────────────────
        writer = (
            deduped_df.write
            .format("delta")
            .option("mergeSchema", "true")
        )

        if load_type == "full" or not folder_exists_dbfs(silver_path):
            writer.mode("overwrite").save(silver_path)
        else:
            # Incremental: upsert via Delta merge on natural key
            if DeltaTable.isDeltaTable(spark, silver_path):
                delta_table = DeltaTable.forPath(spark, silver_path)
                (
                    delta_table.alias("target")
                    .merge(
                        deduped_df.alias("source"),
                        f"target.{natural_key} = source.{natural_key}"
                    )
                    .whenMatchedUpdateAll()
                    .whenNotMatchedInsertAll()
                    .execute()
                )
            else:
                # First run for this entity — create the table
                writer.mode("overwrite").save(silver_path)

        result["silver_rows"] = deduped_df.count()
        result["status"]      = "succeeded"

    except Exception as e:
        result["error"] = str(e)

    return result


print("transform_entity() defined")

In [ ]:
# Cell 7 — Run transformation for all 17 entities sequentially
# (PySpark already parallelises reads/writes internally — sequential Python loop is fine)

print(f"Starting Silver transformation — load_type={LOAD_TYPE}")
print("-" * 60)

results = []
for entity_name, config in ENTITY_REGISTRY.items():
    date_filter = INGESTION_DATE if LOAD_TYPE == "incremental" else ""
    print(f"  Processing: {entity_name} ...", end=" ")
    r = transform_entity(entity_name, config, LOAD_TYPE, date_filter)
    results.append(r)
    if r["status"] == "succeeded":
        print(f"OK  ({r['bronze_rows']} bronze rows -> {r['silver_rows']} silver rows)")
    elif r["status"] == "skipped":
        print(f"SKIP  ({r['error']})")
    else:
        print(f"FAILED  ({r['error']})")

print("-" * 60)
print("All entities processed")

In [ ]:
# Cell 8 — Run summary

succeeded = [r for r in results if r["status"] == "succeeded"]
skipped   = [r for r in results if r["status"] == "skipped"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 70)
print("SILVER API TRANSFORMATION — RUN SUMMARY")
print("=" * 70)
print(f"  load_type      : {LOAD_TYPE}")
print(f"  ingestion_date : {INGESTION_DATE or '(all partitions)'}")
print(f"  run_timestamp  : {RUN_TS}")
print(f"  entities total : {len(results)}")
print(f"  succeeded      : {len(succeeded)}")
print(f"  skipped        : {len(skipped)}")
print(f"  failed         : {len(failed)}")
print()

for r in results:
    if r["status"] == "succeeded":
        tag = "[OK]  "
    elif r["status"] == "skipped":
        tag = "[SKIP]"
    else:
        tag = "[FAIL]"
    print(f"  {tag} {r['entity_name']:<25} {r['status']:<10}  bronze={r['bronze_rows']:>6}  silver={r['silver_rows']:>6}")
    if r["status"] == "failed":
        print(f"          Error: {r['error']}")

print()
if failed:
    raise Exception(f"{len(failed)} entity(ies) failed: {', '.join(r['entity_name'] for r in failed)} — check output above.")
else:
    print(f"Silver transformation complete. {len(succeeded)} entities written to {SILVER_API_BASE}")

In [ ]:
# Cell 9 — Verify Silver Delta tables (spot-check 3 entities)
# Run this after the transformation to confirm Delta tables were created correctly.

spot_check_entities = ["payments", "sessions", "customers"]

print("SILVER VERIFICATION — spot check")
print("-" * 50)

for entity in spot_check_entities:
    silver_path = f"{SILVER_API_BASE}/{entity}"
    if not folder_exists_dbfs(silver_path):
        print(f"  {entity}: NOT FOUND at {silver_path}")
        continue
    try:
        df = spark.read.format("delta").load(silver_path)
        count = df.count()
        cols  = len(df.columns)
        print(f"  {entity:<25} rows={count:>6}  cols={cols}")
        # Show schema for first entity only
        if entity == spot_check_entities[0]:
            print()
            df.printSchema()
            df.show(3, truncate=True)
    except Exception as e:
        print(f"  {entity}: READ ERROR — {e}")